# Best PainNAS architecture: full 87-subject LOSO

This notebook evaluates one fixed architecture with a complete leave-one-subject-out (LOSO) experiment. For each of the 87 BioVid subjects, it creates a **new model and optimizer**, estimates normalization from the other 86 subjects, trains using only those source subjects, and evaluates the held-out subject once. No weights are transferred between folds.

> This produces a target-independent estimate conditional on the fixed architecture. Because the architecture below was chosen after cohort-level NAS, the result remains exploratory with respect to architecture selection unless that choice was pre-specified.

## 1. Mount Drive and check out the repository

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import sys

REPO_URL = 'https://github.com/hhihn/FewShotPainAdaptation.git'
PROJECT_DIR = Path('/content/FewShotPainAdaptation')
BRANCH_NAME = 'painnas'

if not PROJECT_DIR.exists():
    !git clone -b $BRANCH_NAME $REPO_URL $PROJECT_DIR
else:
    %cd $PROJECT_DIR
    !git pull --ff-only
%cd $PROJECT_DIR

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
assert (PROJECT_DIR / 'painnas/loso.py').is_file()

## 2. Install dependencies

In [ ]:
!pip -q install -U pip
!pip -q install -r $PROJECT_DIR/painnas/requirements-colab.txt

## 3. Stage BioVid on the Colab SSD

The dataset is staged from `MyDrive/PainData` to local Colab storage for faster training. Completed folds remain on Drive.

In [ ]:
from data_loaders.dataset_staging import stage_predefined_dataset_from_archive

DRIVE_DATA_DIR = Path('/content/drive/MyDrive/PainData')
LOCAL_DATA_DIR = Path('/content/PainData')
BIOVID_ROOT = stage_predefined_dataset_from_archive(
    'biovid_part_a',
    drive_data_dir=DRIVE_DATA_DIR,
    local_data_dir=LOCAL_DATA_DIR,
    local_archive_dir=Path('/content'),
)
DATA_DIR = LOCAL_DATA_DIR
print('BioVid root:', BIOVID_ROOT)

## 4. Verify the GPU and configure reproducibility

In [ ]:
import random
import numpy as np
import tensorflow as tf

GPUS = tf.config.list_physical_devices('GPU')
assert GPUS, 'Select Runtime > Change runtime type > GPU before continuing.'
for gpu in GPUS:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass

SEED = 42
tf.keras.utils.set_random_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
print('TensorFlow:', tf.__version__, 'GPUs:', GPUS)

## 5. Configure the architecture and LOSO training

The architecture values are the winner from block 1. Change `RUN_NAME` whenever any architecture or training value changes; the saved manifest rejects incompatible resumes.

In [ ]:
from painnas.config import PainNASConfig
from painnas.model import ArchitectureSpec

RUN_NAME = 'block1_best_full_loso'
RESUME = True
VERBOSE = 1

# Training knobs.
BATCH_SIZE = 128
MAX_EPOCHS = 100
PATIENCE = 15
BOOTSTRAP_SAMPLES = 10_000
MAX_PARAMETERS = 25_000_000

# Fixed architecture: block-1 NAS winner.
ARCHITECTURE = ArchitectureSpec(
    num_blocks=4,
    conv_repeats=(2, 2, 2, 2),
    width_multiplier=2.0,
    temporal_kernel_size=15,
    dense_units=(512,),
    learning_rate=0.0007886186679381547,
    dropout_rate=0.25,
    head_type='flatten',
    convolution_type='separable',
    normalization_type='group',
    pooling_type='average',
    pooling_size=4,
)

CONFIG = PainNASConfig(
    seed=SEED,
    batch_size=BATCH_SIZE,
    loso_max_epochs=MAX_EPOCHS,
    loso_patience=PATIENCE,
    bootstrap_samples=BOOTSTRAP_SAMPLES,
    max_parameters=MAX_PARAMETERS,
)
OUTPUT_DIR = Path('/content/drive/MyDrive/PainNAS') / RUN_NAME / 'loso'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(CONFIG)
print(ARCHITECTURE)
print('Output:', OUTPUT_DIR)

## 6. Build and inspect the model

This preview verifies the architecture only. The preview model is deleted; every LOSO fold later constructs another completely fresh model.

In [ ]:
from painnas.model import build_early_fusion_model

preview_model = build_early_fusion_model(ARCHITECTURE)
preview_model.summary()
PARAMETER_COUNT = int(preview_model.count_params())
assert PARAMETER_COUNT == 836_320, PARAMETER_COUNT
assert PARAMETER_COUNT <= CONFIG.max_parameters
print('Parameters:', f'{PARAMETER_COUNT:,}')
del preview_model
tf.keras.backend.clear_session()

## 7. Load BioVid and audit all planned folds

In [ ]:
from painnas.data import build_loso_fold_indices, load_biovid_binary

ARRAYS = load_biovid_binary(str(DATA_DIR), CONFIG)
subjects = [int(value) for value in sorted(ARRAYS.unique_subjects.tolist())]
assert len(subjects) == 87

audit_rows = []
for fold_index, target in enumerate(subjects, start=1):
    fold = build_loso_fold_indices(ARRAYS, target)
    train_subjects = set(map(int, ARRAYS.subjects[fold.train]))
    validation_subjects = set(map(int, ARRAYS.subjects[fold.validation]))
    test_subjects = set(map(int, ARRAYS.subjects[fold.test]))
    checks = {
        'train_excludes_target': target not in train_subjects,
        'validation_excludes_target': target not in validation_subjects,
        'test_contains_only_target': test_subjects == {target},
        'source_subject_count': len(fold.source_subjects) == 86,
    }
    assert all(checks.values()), (fold_index, target, checks)
    audit_rows.append({
        'fold': fold_index,
        'target': target,
        'target_key': ARRAYS.subject_keys.get(target, str(target)),
        'train_samples': len(fold.train),
        'validation_samples': len(fold.validation),
        'test_samples': len(fold.test),
        **checks,
    })

import pandas as pd
from IPython.display import display
audit = pd.DataFrame(audit_rows)
display(audit.head())
print('Audited target-exclusive folds:', len(audit))

## 8. Run or resume all 87 LOSO folds

For every target, `run_loso` clears Keras state, resets the fold seed, constructs a new model, creates a new optimizer, trains on the 86 source subjects, restores the best source-validation weights, and evaluates the target. A completed fold is saved immediately to Drive. Rerun this cell after a Colab disconnect to continue.

In [ ]:
from painnas.loso import run_loso

SUMMARY = run_loso(
    ARRAYS,
    ARCHITECTURE,
    CONFIG,
    OUTPUT_DIR,
    resume=RESUME,
    start_index=None,
    stop_index=None,
    max_folds=None,
    verbose=VERBOSE,
)
display(pd.DataFrame(SUMMARY['metrics']).T)

## 9. Audit the completed experiment

Run this cell after all folds finish. It verifies that each subject was held out once and every fold began with a fresh optimizer.

In [ ]:
import json

result_paths = sorted((OUTPUT_DIR / 'folds').glob('fold_*.json'))
assert len(result_paths) == 87, f'Expected 87 completed folds, found {len(result_paths)}'
results = [json.loads(path.read_text()) for path in result_paths]
targets = [int(result['target_subject']) for result in results]
fingerprints = {result['architecture_fingerprint'] for result in results}
parameter_counts = {int(result['parameter_count']) for result in results}

assert len(set(targets)) == 87
assert set(targets) == set(subjects)
assert all(int(result['source_subject_count']) == 86 for result in results)
assert all(int(result['optimizer_initial_iterations']) == 0 for result in results)
assert parameter_counts == {PARAMETER_COUNT}
assert len(fingerprints) == 1
assert int(SUMMARY['completed_folds']) == 87

audit_summary = pd.DataFrame([{
    'completed_folds': len(results),
    'unique_targets': len(set(targets)),
    'source_subjects_per_fold': 86,
    'fresh_optimizers': all(int(r['optimizer_initial_iterations']) == 0 for r in results),
    'architecture_fingerprints': len(fingerprints),
    'parameter_count': next(iter(parameter_counts)),
}])
display(audit_summary)
print('Full LOSO audit passed.')

## 10. Inspect aggregate and per-subject results

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

fold_metrics = pd.read_csv(OUTPUT_DIR / 'fold_metrics.csv').sort_values('fold_index')
display(fold_metrics)
display(pd.DataFrame(SUMMARY['metrics']).T)

ax = fold_metrics.plot(
    x='fold_index',
    y=['accuracy', 'macro_f1'],
    marker='o',
    markersize=3,
    figsize=(12, 4),
    ylim=(0, 1),
    title='Fresh-model LOSO performance by held-out subject',
)
ax.set_xlabel('LOSO fold')
ax.set_ylabel('Score')
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
plt.show()

confusion = np.asarray(SUMMARY['aggregate_confusion_matrix'], dtype=int)
fig, ax = plt.subplots(figsize=(4, 4))
image = ax.imshow(confusion, cmap='Blues')
for row in range(2):
    for column in range(2):
        ax.text(column, row, f'{confusion[row, column]:,}', ha='center', va='center')
ax.set(
    xticks=(0, 1), yticklabels=('T0', 'T4'),
    yticks=(0, 1), xticklabels=('T0', 'T4'),
    xlabel='Predicted class', ylabel='True class',
    title='Aggregate confusion matrix',
)
fig.colorbar(image, ax=ax, label='Samples')
plt.show()

## 11. Optional runtime cleanup

In [ ]:
import gc
tf.keras.backend.clear_session()
gc.collect()
try:
    from google.colab import runtime
    runtime.unassign()
except ImportError:
    print('Cleanup complete')